In [47]:
from sklearn.decomposition import PCA
from scipy.io import loadmat
import numpy as np
from scipy.spatial.distance import cdist
from scipy.sparse import csr_array
from scipy.sparse.csgraph import shortest_path


# -----------------------------------------------------------------------------
# NOTE: Do not change the parameters / return types for pre defined methods.
# -----------------------------------------------------------------------------
class OrderOfFaces:
    """
    This class handles loading and processing facial image data for dimensionality
    reduction using the ISOMAP algorithm, with PCA as an optional comparison.

    Attributes:
    ----------
    images_path : str
        Path to the .mat file containing the image dataset.

    Methods:
    -------
    get_adjacency_matrix(epsilon):
        Returns the adjacency matrix based on a given epsilon neighborhood.

    get_best_epsilon():
        Returns the best epsilon for the ISOMAP algorithm, likely based on
        graph connectivity or reconstruction error.

    isomap(epsilon):
        Computes a 2D embedding of the data using the ISOMAP algorithm.

    pca(num_dim):
        Returns a low-dimensional embedding of the data using PCA.
    """

    def __init__(self, images_path="data/isomap.mat"):
        """
        Initializes the OrderOfFaces object and loads image data from the given path.

        Parameters:
        ----------
        images_path : str
            Path to the .mat file containing the facial images dataset.
        """

        self.datas = loadmat(images_path)
        data_array = np.array(self.datas["images"])
        data_array_t = data_array.T
        self.data_array_t=data_array_t,
        data_norm = np.linalg.norm(data_array_t, axis=1, keepdims=True)
        data_normalized = data_array_t / data_norm
        self.data_norm = data_normalized
        self.distance = None
        self.adjancy_matrix_viz = None
        self.adjancy_matrix = None

    def get_adjacency_matrix(self, epsilon: float) -> np.ndarray:
        """
        Constructs the adjacency matrix using epsilon neighborhoods.

        Parameters:
        ----------
        epsilon : float
            The neighborhood radius within which points are considered connected.

        Returns:
        -------
        np.ndarray
            A 2D adjacency matrix (m x m) where each entry represents distance between
            neighbors within the epsilon threshold.
        """

        distance_matrix = cdist(self.data_norm, self.data_norm, metric="euclidean")
        adjancy_matrix = np.where(distance_matrix <= epsilon, distance_matrix, 0)
     
        self.distance=distance_matrix
        adjancy_matrix_viz = np.where(distance_matrix <= epsilon, 1, 0)
        self.adjancy_matrix_viz = adjancy_matrix_viz
        self.adjancy_matrix = adjancy_matrix
        return adjancy_matrix

    def get_best_epsilon(self) -> float:
        """
        Heuristically determines the best epsilon value for graph connectivity in ISOMAP.

        Returns:
        -------
        float
            Optimal epsilon value ensuring a well-connected neighborhood graph.


        """

        return 0.409297
        raise NotImplementedError("Not Implemented")

    def isomap(self, epsilon: float) -> np.ndarray:
        """
        Applies the ISOMAP algorithm to compute a 2D low-dimensional embedding of the dataset.

        Parameters:
        ----------
        epsilon : float
            The neighborhood radius for building the adjacency graph.

        Returns:
        -------
        np.ndarray
            A (m x 2) array where each row is a 2D embedding of the original data point.


        """

        adj_matrix = self.get_adjacency_matrix(epsilon)

        graph = csr_array(adj_matrix)
        gdm = shortest_path(csgraph=graph, directed=False, method="FW")
        I = np.identity(gdm.shape[0])
        ones = np.ones((gdm.shape[0], 1))
        H = I - (1 / gdm.shape[0]) * (ones @ ones.T)
        G = (-1 / 2) * H @ (gdm**2) @ H
        e_vals, e_vectors = np.linalg.eigh(G)
        e_vals_sqrt = np.sqrt(e_vals)
        Z = e_vectors[:, -2:] * e_vals_sqrt[-2:]

        return Z

    def pca(self, num_dim: int) -> np.ndarray:
        """
        Applies PCA to reduce the dataset to a specified number of dimensions.

        Parameters:
        ----------
        num_dim : int
            Number of principal components to project the data onto.

        Returns:
        -------
        np.ndarray
            A (m x num_dim) array representing the dataset in a reduced PCA space.


        """
        pca = PCA(n_components=num_dim)

        pca_embedding = pca.fit_transform(self.data_norm)
        return pca_embedding



In [48]:
test = OrderOfFaces()

In [49]:
test.get_adjacency_matrix(.45)

array([[0.        , 0.        , 0.20865425, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.20865425, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(698, 698))

In [52]:
test.distance[0,2]

np.float64(0.20865425229620538)

In [31]:
n = test.data_norm

In [71]:
X = n
squared_norms = np.sum(X**2, axis=1, keepdims=True, dtype=np.float64)
distance_matrix_sq = squared_norms + squared_norms.T - 2 * X @ X.T
distance_matrix_sq = np.maximum(distance_matrix_sq, 0)
distance_matrix = np.sqrt(distance_matrix_sq, dtype=np.float64)

In [77]:
np.where(distance_matrix <= .5453465, distance_matrix, 0)

array([[9.42432183e-08, 0.00000000e+00, 2.08654252e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 5.37269007e-08, 0.00000000e+00, ...,
        5.16054808e-01, 0.00000000e+00, 0.00000000e+00],
       [2.08654252e-01, 0.00000000e+00, 6.66400187e-08, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 5.16054808e-01, 0.00000000e+00, ...,
        1.04308128e-07, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]], shape=(698, 698))

In [76]:
cdist(X, X, metric="euclidean").astype(np.float64)

array([[0.        , 0.64944326, 0.20865425, ..., 0.67281836, 0.72142858,
        0.62043311],
       [0.64944326, 0.        , 0.64140337, ..., 0.51605481, 0.71196516,
        0.84348776],
       [0.20865425, 0.64140337, 0.        , ..., 0.65932819, 0.71915014,
        0.60405182],
       ...,
       [0.67281836, 0.51605481, 0.65932819, ..., 0.        , 0.6799211 ,
        0.76904231],
       [0.72142858, 0.71196516, 0.71915014, ..., 0.6799211 , 0.        ,
        0.66191604],
       [0.62043311, 0.84348776, 0.60405182, ..., 0.76904231, 0.66191604,
        0.        ]], shape=(698, 698))

In [68]:
adjancy_matrix = np.where(distance_matrix <= .45452347335, distance_matrix, 0)

In [69]:
adjancy_matrix

array([[9.42432183e-08, 0.00000000e+00, 2.08654252e-01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 5.37269007e-08, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [2.08654252e-01, 0.00000000e+00, 6.66400187e-08, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        1.04308128e-07, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]], shape=(698, 698))

In [53]:
squared_diffs = (n[0] - n[2]) ** 2
sum_of_squares = np.sum(squared_diffs)
distance_b = np.sqrt(sum_of_squares)

In [54]:
distance_b

np.float64(0.20865425229620568)

In [26]:
a = test.get_adjacency_matrix(.0001)

In [ ]:
test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(698, 698))

In [36]:
cdist(n[0],n[2])

ValueError: XA must be a 2-dimensional array.

In [19]:
np.sqrt(a)

array([[0.        , 0.        , 0.45678688, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.71836955, 0.        ,
        0.        ],
       [0.45678688, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.71836955, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(698, 698))